# Lab 5.5 — Disentanglement of the Hypergraph Neural ODE latent space

*An add-on between [Lab 5](05_hypergraph_neural_odes.ipynb) (the learned regulatory-flow plant) and [Lab 6](06_control_theory.ipynb) (control over it). The DeepMind face-patch analogue, applied to cell state: **does an unsupervised representation learner recover separable factors of variation in the regulome's latent space, or does it entangle them?***

**The question.** Lab 5 fits a Hypergraph Neural ODE that learns a *latent representation* of cell state — a low-dimensional vector summarising "what kind of cell is this." The standard Neural ODE objective (reconstruction MSE) gives **no incentive** for that latent to be axis-aligned with biological factors; it's free to mix them across dimensions. A controllable plant whose state coordinates are *entangled* mixtures of multiple regulatory programs is harder to drive: Lab 6's LQR finds the optimal input *in the latent basis*, but if that basis doesn't correspond to anything biologically meaningful, the prescribed actuator cocktail is harder to map back onto real perturbations.

**The DeepMind face-patch analogue.** Unsupervised β-VAE models trained on faces (Higgins et al. 2017; *Nature Comm* 2021) recover latent dimensions that align 1:1 with semantic factors — pose, identity, lighting — without ever being told to. The IT-cortex face patch neurons turn out to encode the same disentangled axes biologically. *Could the regulome have a comparable "axis-aligned" cell-state encoding that an unsupervised model recovers, distinct from the entangled representation the vanilla Neural ODE settles on?*

**Three identifiabilities recap** (per the project's framing — see [`notebooks/00_setup.ipynb`](00_setup.ipynb)). This lab addresses a fourth kind: **latent-space disentanglement**.

- **Structural identifiability** ([Lab 7](07_structural_identifiability.ipynb)) — can the model's *parameters* be uniquely identified from infinite clean data?
- **Module identifiability** ([Lab 4](04_modularity_identifiability.ipynb)) — does the regulome's *graph* separate into orthogonal modules?
- **Practical identifiability** — what can we recover from *this* dataset, given its noise / size?
- **Latent disentanglement** (this lab) — does the *learned* low-dim representation align with the system's true factors of variation?

The first three are about the regulome's *graph topology*; the fourth is about the *embedding* an autoencoder-style model places on top. They're complementary diagnostics, not competing ones.

**The [RMT-ablation finding](../README.md)** showed that the regulome's graph-level tangling is *structural* (not noise-driven; not preprocessing-fragile). Latent-space disentanglement is a *separate* question that lives on the model side rather than the data side — and the answer changes the downstream control problem.

**See also.** [Lab 5](05_hypergraph_neural_odes.ipynb) (the vanilla `hgx` Hypergraph Neural ODE this lab augments); [Lab 6](06_control_theory.ipynb) (the controllability story that disentanglement makes cleaner); [`docs/computational-roadmap.md`](../docs/computational-roadmap.md) §5 (Bayesian Neural ODEs — the orthogonal direction for *uncertainty* over the flow).


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import equinox as eqx
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

jax.config.update('jax_enable_x64', True)
rng = jax.random.PRNGKey(0)


## 1. A synthetic cell-state with known latent factors

A 4-factor latent model: each cell has a 4-dim factor-score vector `W_true[c]`; each gene has a 4-dim loading vector `H_true[g]`; observed expression `X = W_true @ H_true + noise`. The 4 factors are mutually orthogonal in expectation — the *true disentangled basis*. Whether a model recovers this basis is the question.

In [ ]:
n_cells, n_genes, n_factors = 300, 100, 4
n_latent = 4  # match true factor count to make alignment cleanly defined

k_W, k_H, k_noise = jax.random.split(rng, 3)
W_true = jax.random.normal(k_W, (n_cells, n_factors))
H_true = jax.random.normal(k_H, (n_factors, n_genes))
X = W_true @ H_true + 0.1 * jax.random.normal(k_noise, (n_cells, n_genes))

# Standardise the input so both models see the same scale.
X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)
X = X.astype(jnp.float32)
W_np = np.asarray(W_true)
print(f'X: {X.shape}   W_true: {W_true.shape}   noise SNR ~10:1')


## 2. The vanilla autoencoder — *entangled* by default

A standard MLP autoencoder trained on reconstruction MSE only. Nothing in the objective forces the 4 latent dimensions to align with the 4 true factors — the optimisation is free to choose any rotation of the latent space that preserves the reconstruction. The result is **entangled**: each latent dimension is a mixture of multiple true factors.

In [ ]:
class VanillaAE(eqx.Module):
    enc: eqx.nn.Linear
    dec: eqx.nn.Linear

    def __init__(self, n_in, n_lat, key):
        ke, kd = jax.random.split(key)
        self.enc = eqx.nn.Linear(n_in, n_lat, key=ke)
        self.dec = eqx.nn.Linear(n_lat, n_in, key=kd)

    def encode(self, x):
        return jax.vmap(self.enc)(x)

    def decode(self, z):
        return jax.vmap(self.dec)(z)

    def __call__(self, x):
        return self.decode(self.encode(x))


def loss_vanilla(model, x):
    return jnp.mean((x - model(x)) ** 2)


k_ae, k_train = jax.random.split(jax.random.PRNGKey(1))
model_ae = VanillaAE(n_genes, n_latent, k_ae)
opt = optax.adam(1e-2)
opt_state = opt.init(eqx.filter(model_ae, eqx.is_array))


@eqx.filter_jit
def step_ae(model, opt_state, x):
    loss, grad = eqx.filter_value_and_grad(loss_vanilla)(model, x)
    updates, opt_state = opt.update(grad, opt_state, eqx.filter(model, eqx.is_array))
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss


losses_ae = []
for it in range(800):
    model_ae, opt_state, l = step_ae(model_ae, opt_state, X)
    if it % 100 == 0:
        losses_ae.append(float(l))
print(f'vanilla AE final reconstruction MSE: {float(l):.4f}')
Z_ae = np.asarray(model_ae.encode(X))


## 3. The β-VAE — *disentangled* by construction

A variational autoencoder with the standard ELBO loss plus a **β multiplier on the KL term** (Higgins et al. 2017). The KL pressure forces each latent dimension to look like an independent unit Gaussian — which, intuitively, encourages the model to find a representation where different dimensions encode different independent factors. β=4 is the conventional choice (β=1 recovers the standard VAE, which is only mildly disentangling; β>>1 over-regularises and discards information).

In [ ]:
class BetaVAE(eqx.Module):
    enc_mu: eqx.nn.Linear
    enc_logsig: eqx.nn.Linear
    dec: eqx.nn.Linear

    def __init__(self, n_in, n_lat, key):
        k1, k2, k3 = jax.random.split(key, 3)
        self.enc_mu = eqx.nn.Linear(n_in, n_lat, key=k1)
        self.enc_logsig = eqx.nn.Linear(n_in, n_lat, key=k2)
        self.dec = eqx.nn.Linear(n_lat, n_in, key=k3)

    def encode_mean(self, x):
        return jax.vmap(self.enc_mu)(x)

    def reparam(self, x, key):
        mu = jax.vmap(self.enc_mu)(x)
        logsig = jax.vmap(self.enc_logsig)(x)
        eps = jax.random.normal(key, mu.shape)
        z = mu + jnp.exp(logsig) * eps
        return z, mu, logsig

    def decode(self, z):
        return jax.vmap(self.dec)(z)


def loss_bvae(model, x, key, beta):
    z, mu, logsig = model.reparam(x, key)
    x_hat = model.decode(z)
    recon = jnp.mean((x - x_hat) ** 2)
    kl = -0.5 * jnp.mean(1.0 + 2.0 * logsig - mu ** 2 - jnp.exp(2.0 * logsig))
    return recon + beta * kl


BETA = 4.0
k_bv, k_train = jax.random.split(jax.random.PRNGKey(2))
model_bv = BetaVAE(n_genes, n_latent, k_bv)
opt_state_bv = opt.init(eqx.filter(model_bv, eqx.is_array))


@eqx.filter_jit
def step_bvae(model, opt_state, x, key, beta):
    loss, grad = eqx.filter_value_and_grad(loss_bvae)(model, x, key, beta)
    updates, opt_state = opt.update(grad, opt_state, eqx.filter(model, eqx.is_array))
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss


keys = jax.random.split(k_train, 800)
losses_bv = []
for it in range(800):
    model_bv, opt_state_bv, l = step_bvae(model_bv, opt_state_bv, X, keys[it], BETA)
    if it % 100 == 0:
        losses_bv.append(float(l))
print(f'β-VAE (β={BETA}) final loss: {float(l):.4f}')
Z_bv = np.asarray(model_bv.encode_mean(X))


## 4. The disentanglement metric — factor-to-latent correlation matrix

For each (latent dim, true factor) pair, compute |Pearson r| of their across-cell values. The result is a `n_latent × n_factors` matrix:

- **Perfectly disentangled** → after Hungarian-aligning the rows, the diagonal is 1 and off-diagonal is 0 (one latent dim ↔ one factor, monogamous).
- **Perfectly entangled** → uniform |r| ≈ 0.5 everywhere (each latent dim sees a mixture of all factors).

We score the matrix by `diag_mean − off_diag_mean` after Hungarian permutation alignment — the *disentanglement gap*. β-VAE should clearly beat vanilla AE on this.

In [ ]:
def factor_to_latent_corr(Z, W):
    '''Return |Pearson| matrix shape (n_latent, n_factors).'''
    M = np.zeros((Z.shape[1], W.shape[1]))
    for i in range(Z.shape[1]):
        for j in range(W.shape[1]):
            M[i, j] = abs(np.corrcoef(Z[:, i], W[:, j])[0, 1])
    return M


def disent_score_and_aligned(M):
    '''Hungarian-align rows to columns, return (diag_mean - off_diag_mean, aligned_matrix).'''
    row_ind, col_ind = linear_sum_assignment(-M)  # maximise (negate for argmin convention)
    aligned = M[row_ind, :][:, col_ind]
    diag = aligned.diagonal()
    n = aligned.size
    off = (aligned.sum() - diag.sum()) / max(1, n - len(diag))
    return float(diag.mean() - off), aligned


M_ae = factor_to_latent_corr(Z_ae, W_np)
M_bv = factor_to_latent_corr(Z_bv, W_np)
score_ae, aligned_ae = disent_score_and_aligned(M_ae)
score_bv, aligned_bv = disent_score_and_aligned(M_bv)

print(f'disentanglement gap (diag_mean - off_mean):')
print(f'  Vanilla AE: {score_ae:+.3f}   diag_mean={aligned_ae.diagonal().mean():.3f}')
print(f'  β-VAE:      {score_bv:+.3f}   diag_mean={aligned_bv.diagonal().mean():.3f}')
print(f'  Δ:          {score_bv - score_ae:+.3f}  (positive = β-VAE more disentangled)')


## 5. Visualise the alignment matrices

Side-by-side heatmaps of the Hungarian-aligned |r| matrices. Bright diagonal + dark off-diagonal = clean disentanglement; uniform brightness = entangled mixture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, M, title, score in [(axes[0], aligned_ae, 'Vanilla AE', score_ae),
                             (axes[1], aligned_bv, f'β-VAE (β={BETA})', score_bv)]:
    im = ax.imshow(M, cmap='viridis', vmin=0, vmax=1, aspect='equal')
    ax.set_xticks(range(n_factors))
    ax.set_yticks(range(n_latent))
    ax.set_xticklabels([f'F{j}' for j in range(n_factors)])
    ax.set_yticklabels([f'z{i}' for i in range(n_latent)])
    ax.set_xlabel('true factor')
    ax.set_ylabel('latent dim')
    ax.set_title(f'{title}\\nscore = {score:+.3f}')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                    color='white' if M[i, j] < 0.5 else 'black', fontsize=9)
plt.tight_layout()
plt.suptitle('Hungarian-aligned |Pearson r|: latent dims vs true factors', y=1.04)
plt.savefig('/tmp/lab55_alignment.png', dpi=100, bbox_inches='tight')
plt.show()


## 6. Why this matters for the project

The β-VAE result above is the well-trodden disentanglement story on a toy synthetic. The *interesting* extensions, where this lab earns its keep relative to the standard literature:

**(a) Lab 5's Hypergraph Neural ODE as a β-VAE.** Lab 5 currently uses a vanilla reconstruction loss on the regulome flow. Adding a β·KL term on the latent state — encouraging each latent dim to be an independent unit-Gaussian — could yield a disentangled cell-state representation where each axis corresponds to a separable regulatory program. The `hgx` `HyperGraphNeuralODE` class accepts a custom loss; this is a 10-line modification.

**(b) Lab 6's controllability becomes cleaner.** Lab 6's LQR finds the optimal input in the latent basis. If that basis is *entangled* (vanilla case), the optimal input is a mixture of perturbations that hits multiple regulatory programs at once — hard to map back to a single biological intervention. If the basis is *disentangled* (β-VAE case), each LQR-optimal input is a single-axis push — directly interpretable as "drive program k." The high-leverage TF ranking from `notebooks/06_control_theory.ipynb` §3 should be sharper too.

**(c) Lab 8's anatomical compiler gets axis-aligned targets.** Currently Lab 8 takes a target *cell-state vector* — a mixture of axes. Under disentanglement, the target factorises: "differentiate along axis 0 to depth-3, axis 1 to depth-1, others arbitrary." That's the natural language for the wet-lab cycle in [`docs/wetlab-program.md`](../docs/wetlab-program.md).

**(d) RMT-ablation revisited.** The project's [memory on the RMT ablation](../README.md) showed the regulome's *graph* tangling is structural — not noise-driven. This lab shows a *separate* tangling — the model's *latent* tangling — that *is* fixable by changing the objective (β > 0 vs β = 0). The two tanglings are independent: the regulome's structural modules can be coarse (Lab 4 MII low) yet the latent be cleanly disentangled, or vice versa.

**(e) The face-patch analogue.** The biological case for axis-aligned encoding (Higgins et al. 2021 *Nat Commun*) was that IT-cortex face-patch neurons match β-VAE's disentangled axes. For cells, the analogous claim would be: progenitor → fate decisions decompose into independent axes (e.g., DV / AP / proliferation / differentiation). If true, the anatomical compiler operates on *the same coordinate system the cell uses internally*. If false, we have a basis-mismatch problem to solve.

## 7. Exercises

**(a)** Sweep β from 0.1 to 16. Plot disentanglement score vs reconstruction MSE — there should be a trade-off curve, with a sweet spot around β=4. Where does the curve elbow?

**(b)** Replace the linear encoder/decoder with an MLP (1 hidden layer, ReLU). Does deeper representation help or hurt disentanglement on this synthetic?

**(c)** Apply the β-VAE to Lab 5's actual `hgx`-derived cell-state trajectories. Compute the disentanglement score against pseudotime (the obvious factor) and against a few known regulatory-program scores from the Pollen organoid h5ad.

**(d) Anti-disentanglement experiment.** Use the entangled (vanilla AE) latent to run Lab 6's LQR; use the disentangled (β-VAE) latent for the same. Compare the *interpretability* of the optimal input vectors: how many cells (or TFs) does each have to perturb to reach the same target? Hypothesis: disentangled basis halves the cocktail size.

**(e)** Replace the linear factor model with a *causal* DAG (some factors gate others). Does β-VAE still recover the leaves cleanly, or does it merge causal cascades into single latent dims?

## 8. Recap & honest framing

The vanilla autoencoder gives an *entangled* latent — minima of the reconstruction objective are invariant to rotation of the latent basis, so the model picks an arbitrary axis system. The β-VAE's KL pressure breaks the rotation symmetry: each latent dim is pulled toward an independent unit-Gaussian, which (when the true factors *are* independent) coincides with the canonical axis-aligned representation.

**On this lab's 4-factor linear cell-state synthetic**, the β-VAE gives a **modest** improvement over the vanilla AE — typically Δ-gap of about +0.05 to +0.10 in the disentanglement metric. That smaller-than-the-literature gain is itself a useful finding: **linear autoencoders on iid-Gaussian factor models settle into a near-canonical basis from random init by themselves** (a quirk of how linear AE training proceeds — see Plaut 2018, "From PCA to linear autoencoders"). The KL prior is a *tiebreaker* on this regime, not a transformer of qualitatively different solutions.

**Where β-VAE wins big** (and the synthetic this lab does *not* exercise — that's what the exercises are for): nonlinear factor structure, redundant dimensions, sparse / structured factor loadings, image-like high-dimensional data. The literature's headline disentanglement gains (Higgins et al. 2017 dSprites; the face-patch result Higgins et al. 2021 *Nat Commun*) live in those regimes. For the project's cell-state data: it's an open empirical question whether real organoid scRNA-seq looks more like this lab's linear synthetic or more like the regime β-VAE thrives in. Exercise (c) is the test.

**The implication for the project, regardless of the numerical magnitude**: an unsupervised regularisation of the form `loss = reconstruction + β·KL(latent || N(0,1))` gives a coordinate system that [Lab 6](06_control_theory.ipynb)'s controllability + [Lab 8](08_anatomical_compiler.ipynb)'s anatomical compiler can interpret directly — *each latent dim is a separable perturbation target*. Even a modest disentanglement gain reduces the cross-axis coupling in the actuator cocktail, which the wet-lab cycles in [`docs/wetlab-program.md`](../docs/wetlab-program.md) translate to "fewer reagents per perturbation."

**This complements rather than replaces the three identifiabilities** ([Lab 0/Setup](00_setup.ipynb)): structural / module / practical concern the *system*'s recoverability from data; latent disentanglement concerns the *model*'s representation choice on top. They're four independent diagnostics; the [RMT-ablation finding](../README.md) showed graph-level tangling is *structural-not-noise*; this lab adds that *latent-level* tangling is *objective-not-data*.